<a href="https://colab.research.google.com/github/lilydeller/Math344-Project/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part A


Import libraries for downloading and working with the data. Certificate verification is skipped to get around the download error and then the related warning messages are hidden. Then it downloads the dataset if it hasn't been downloaded before. Then the zip file is extracted. Then the ratings are put into a table with user, item, rating, and time columns. Then displays the first 5 rows and the number of ratings.

In [1]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import requests

In [2]:
ML_DATASET = Path("ml-100k")

if not ML_DATASET.exists():
    url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
    response = requests.get(url, verify=False)

    ZipFile(BytesIO(response.content)).extractall()

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'files.grouplens.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [3]:
movie_ratings = pd.read_csv(
    ML_DATASET / "u.data", sep="\t", names=["user", "item", "rating", "time"]
)

print(movie_ratings.head())
print("Number of ratings:", len(movie_ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


# Part B

A function that split the ratings into training data and testing data by randomly selecting ratings without replacement through using a permutation of the positions of the ratings. It takes in a percentage of ratings where it is probability of a rating to be selected as test data, and it returns array of Booleans where if the rating data at that position is testing data, then it is true, otherwise false. And the function also returns the testing data array inverted to create the positions of ratings that is training data. If a rating is selected as test data and the user and movie of that rating has at least two ratings in the training data, then mark the rating at that position as test data and decrement the amount of ratings that the user and movie has for the training data. It also takes a seed for the random number generator to create the permutation and select test data. Then split the movie ratings into training data and testing data using the default test fraction of 0.10 and seed 0.

In [4]:
def split_ratings(ratings: pd.DataFrame, test_fraction=0.10, seed=0):
    # Hide ratings without removing a user's or movie's last training rating.
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(ratings), dtype=np.bool)

    user_left = ratings.groupby("user").size().to_dict()
    item_left = ratings.groupby("item").size().to_dict()

    for pos in rng.permutation(len(ratings)):
        row = ratings.iloc[pos]
        user, item = row["user"], row["item"]

        if rng.random() < test_fraction and user_left[user] > 1 and item_left[item] > 1:
            is_test[pos] = True
            user_left[user] -= 1
            item_left[item] -= 1

    return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

In [5]:
train, test = split_ratings(movie_ratings)
print("train: ", len(train), "test: ", len(test))
print("test fraction: ", round(len(test) / len(movie_ratings), 3))

train:  90032 test:  9968
test fraction:  0.1


# Part C

Set the number of users and movies in the dataset. Defines a function that creates a user-by-movie matrix filled with missing values. Inserts training ratings at its user row and movie column. User and movie IDs start at 1, but row and column positions in python start counting at 0 so 1 is subtracted from user and item. Then it creates a Boolean array identifying the known training ratings. Then displays the matrix dimensions and the number of know training ratings.

In [6]:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
  R = np.full((N_USERS, N_ITEMS), np.nan)
  R[df["user"].to_numpy()- 1,
    df["item"].to_numpy()- 1] = df["rating"].to_numpy()
  return R

R = to_matrix(train)
known = ~np.isnan(R)

print("Matrix shape:", R.shape)
print("Known training ratings:", known.sum())

Matrix shape: (943, 1682)
Known training ratings: 90032


This stores the actual test ratings into a variable called "truth" so we can compare them with our predictions. Also stores the user row and movie column for each test rating.

In [7]:
truth = test["rating"].to_numpy()
u = test["user"].to_numpy()- 1
i = test["item"].to_numpy()- 1

# Part D

The RMSE first finds the prediction error, how far off each prediction was from its corresponding actual rating, then squares it. Then it finds the average prediction error and returns the square root of that average. The returned value is how far off the prediction is on average, in stars, from the actual ratings.

In [9]:
def rmse(pred, truth):
  return np.sqrt(np.mean((pred - truth)**2))

# Part E

np.nanmean(R) gets the average of the training ratings that are not nan, and assigns that value to mu. The user mean retrieves the average rating of each user, since axis=1 takes the average of each row. n_item is the number of ratings in each column (for each movie) and the item mean starts with each movie's predicted rating as the overall average (mu), then, if the movie has a rating, the average for that movie is replaced with the average of its ratings. If a movie has no ratings, the rating remains as the overall average.

In [12]:
mu = np.nanmean(R)
user_mean = np.nanmean(R, axis=1)

n_item = known.sum(axis=0)
item_mean = np.full(N_ITEMS, mu)
has_rating = n_item > 0
item_mean[has_rating] = (np.nansum(R, axis=0)[has_rating] / n_item[has_rating])

print("Overall average:", round(mu, 3))

Overall average: 3.53


This code finds the rmse for four ways of predicting movie ratings using the rmse function defined in part D. The first makes a prediction of 3 stars for each hidden rating. The second makes a prediction with the overall average for each hidden rating. User average predicts the hidden ratings using each user's average training rating and predicts that rating for every movie. Movie average predicts the hidden ratings of each movie by using the average of the movie's training ratings and predicts that for every user. The movie average performs best with the lowest rmse.

In [13]:
scores = {}
scores["always 3"] = rmse(np.full(len(test), 3.0), truth)
scores["overall average"] = rmse(np.full(len(test), mu), truth)
scores["user average"] = rmse(user_mean[u], truth)
scores["movie rating"] = rmse(item_mean[i], truth)

for name, score in scores.items():
  print(f"{name:16s} {score:.4f}")

always 3         1.2516
overall average  1.1349
user average     1.0426
movie rating     1.0284


# Part F

# Practice Exercises

## Exercise 1

## Exercise 2

## Exercise 3